# 🚀 MoEBoost on Google Colab (with Real-time Prefill & Generation tok/s)

Welcome to **MoEBoost**! This notebook packages and sets up the complete MoEBoost engine directly in Google Colab.

### Features included:
1. **Full MoEBoost Engine setup** (CUDA-accelerated llama.cpp, custom VRAM/RAM workstation paging, C++ high-performance paging kernel).
2. **Direct Python Model Loader**: Load GGUF models directly via Python and run text-based chat in your notebook.
3. **Real-time Performance Metrics**: Displays **Prefill tok/s**, **Generation tok/s**, **Time-To-First-Token (TTFT)**, and Wall clock duration for every turn.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Install Dependencies & Build CUDA llama-cpp-python

In [ ]:
# Install core python requirements
!pip install --upgrade pip
!pip install fastapi uvicorn psutil numpy pydantic huggingface_hub pybind11 cmake

# Build CUDA-accelerated llama-cpp-python for Google Colab GPU (T4 / V100 / A100 / L4)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install --no-cache-dir llama-cpp-python

## 3. Unpack MoEBoost Source & Compile C++ Pager Extension

In [ ]:
import os, base64

# Read embedded MoEBoost archive
b64_path = 'moeboost_source.tar.gz.b64'
if os.path.exists(b64_path):
    with open(b64_path, 'r') as f:
        data = base64.b64decode(f.read())
    with open('moeboost_source.tar.gz', 'wb') as f:
        f.write(data)
    !tar -xzf moeboost_source.tar.gz -C .
    print('✅ MoEBoost source unpacked successfully!')
else:
    print('ℹ️ If running locally or from uploaded files, ensure MoEBoost files are present in the directory.')

# Compile high-performance C++ MoE pager extension if CMakeLists exists
if os.path.exists('CMakeLists.txt'):
    !mkdir -p build && cd build && cmake .. && make -j$(nproc)
    !cp build/*.so . 2>/dev/null || true
    print('✅ C++ Pager core ready!')

## 4. Built-in Hugging Face MoE Model Catalog (One-Click Download)
Choose from curated MoE models (Qwen-MoE, DeepSeek-MoE, Mixtral-8x7B, Qwen2.5-Coder) with interactive Colab dropdown forms, or enter your own custom Hugging Face model link.

In [ ]:
# @title 📥 Select & Download MoE GGUF Model from Hugging Face { run: "auto" }
# @markdown Select one of the curated popular MoE models below, or choose "Custom" to enter your own:

Model_Preset = "Qwen2.5-Coder-7B (Fast Developer Baseline)" # @param ["Qwen2.5-Coder-7B (Fast Developer Baseline)", "Qwen1.5-MoE-A2.7B (Ultra-Fast 4x2.7B MoE)", "Mixtral-8x7B-Instruct (Standard 8x7B MoE)", "DeepSeek-V2-Lite-Chat (16B MoE)", "Llama-3.2-1B-Instruct (Tiny Edge Baseline)", "Custom"]

custom_repo_id = "" # @param {type:"string"}
custom_filename = "" # @param {type:"string"}

from huggingface_hub import hf_hub_download
import os

MOE_CATALOG = {
    "Qwen2.5-Coder-7B (Fast Developer Baseline)": {
        "repo_id": "Qwen/Qwen2.5-Coder-7B-Instruct-GGUF",
        "filename": "qwen2.5-coder-7b-instruct-q4_k_m.gguf",
        "desc": "High quality 7B coding model, fast and fits comfortably on standard Colab T4 (15GB VRAM)."
    },
    "Qwen1.5-MoE-A2.7B (Ultra-Fast 4x2.7B MoE)": {
        "repo_id": "Qwen/Qwen1.5-MoE-A2.7B-Chat-GGUF",
        "filename": "qwen1_5-moe-a2_7b-chat-q4_k_m.gguf",
        "desc": "Native Mixture-of-Experts (14.3B total, 2.7B active). High inference speed."
    },
    "Mixtral-8x7B-Instruct (Standard 8x7B MoE)": {
        "repo_id": "TheBloke/Mixtral-8x7B-Instruct-v0.1-GGUF",
        "filename": "mixtral-8x7b-instruct-v0.1.Q3_K_M.gguf",
        "desc": "Iconic 8x7B MoE (46.7B total, 12.9B active). Recommended for High-RAM / A100 Colab runtimes."
    },
    "DeepSeek-V2-Lite-Chat (16B MoE)": {
        "repo_id": "bartowski/DeepSeek-V2-Lite-Chat-GGUF",
        "filename": "DeepSeek-V2-Lite-Chat-Q4_K_M.gguf",
        "desc": "DeepSeek MoE architecture with fine-grained experts (16B total, 2.4B active)."
    },
    "Llama-3.2-1B-Instruct (Tiny Edge Baseline)": {
        "repo_id": "bartowski/Llama-3.2-1B-Instruct-GGUF",
        "filename": "Llama-3.2-1B-Instruct-Q4_K_M.gguf",
        "desc": "Ultra lightweight 1B model for rapid testing and low memory overhead."
    }
}

if Model_Preset == "Custom":
    repo_id = custom_repo_id.strip()
    filename = custom_filename.strip()
    if not repo_id or not filename:
        raise ValueError("Please provide custom_repo_id and custom_filename!")
else:
    info = MOE_CATALOG[Model_Preset]
    repo_id = info["repo_id"]
    filename = info["filename"]
    print(f"📖 Selected: {Model_Preset}")
    print(f"ℹ️ Description: {info['desc']}")

print(f"\n🚀 Downloading '{filename}' from Hugging Face: {repo_id}...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"\n✅ Model ready at: {model_path}")
print(f"   Size: {os.path.getsize(model_path) / (1024**3):.2f} GB")


## 5. Direct Python MoEBoost Model Loader & Interactive Chat
Run interactive inference with **real-time Prefill and Generation tok/s** displayed on every turn!

In [ ]:
import sys, os, time
from pathlib import Path
from engine.runtime import LlamaCppRuntime

def start_chat(model_file_path: str, gpu_layers: int = -1):
    runtime = LlamaCppRuntime()
    runtime._settings["context_size"] = 4096
    runtime._settings["batch_size"] = 512
    runtime._settings["requested_gpu_layers"] = gpu_layers

    print(f"⚡ Loading GGUF Model into MoEBoost: {os.path.basename(model_file_path)}")
    t0 = time.perf_counter()
    status = runtime.load(model_file_path, max_safe_gpu_layers=gpu_layers, backend="cuda")
    t1 = time.perf_counter()
    print(f"✅ Model loaded in {t1 - t0:.2f}s | Active GPU Layers: {status.get('gpu_layers', 'auto')}")

    messages = [{"role": "system", "content": "You are a helpful and intelligent AI assistant powered by MoEBoost."}]
    print("\n=== Interactive MoEBoost Chat Started (Type 'exit' or 'quit' to end) ===\n")

    while True:
        user_input = input("👤 User: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit"):
            print("Exiting session. Unloading model...")
            runtime.unload()
            break

        messages.append({"role": "user", "content": user_input})
        print("\n🤖 Assistant (generating...): ", end="", flush=True)

        t0 = time.perf_counter()
        result = runtime.completion(messages=messages, temperature=0.7, max_tokens=512, stream=False)
        t1 = time.perf_counter()
        wall_time = max(0.001, t1 - t0)

        choices = result.get("choices") or []
        reply = (choices[0].get("message") or {}).get("content", "") if choices else ""
        print(reply)

        # Extract precise MoEBoost performance metrics
        perf = result.get("performance") or runtime._read_perf_counters() or {}
        gen_tok_sec = perf.get("generation_tokens_per_second")
        prefill_tok_sec = perf.get("prefill_tokens_per_second")
        ttft = perf.get("last_time_to_first_token_formatted")

        usage = result.get("usage") or {}
        completion_tokens = usage.get("completion_tokens", 0)
        if not gen_tok_sec and completion_tokens:
            gen_tok_sec = round(completion_tokens / wall_time, 2)

        print("\n" + "-" * 65)
        print(f"⚡ [Stats] Generation: {gen_tok_sec or 'N/A'} tok/s | Prefill: {prefill_tok_sec or 'N/A'} tok/s | TTFT: {ttft or 'N/A'} | Wall: {wall_time:.2f}s")
        print("-" * 65 + "\n")
        messages.append({"role": "assistant", "content": reply})

# Start chat with downloaded model
start_chat(model_path, gpu_layers=-1)